# Notebook 04 — Benchmarks and Evaluation

**What you’ll learn:** How to define **benchmark questions** for your Genie space — questions with known SQL answers that let you measure accuracy over time.

**Why this matters:** When you change instructions, add examples, or Databricks updates the model, benchmarks tell you whether Genie got better or worse. Think of them as **unit tests for your Genie space**.

**What this notebook does:**
1. Defines 16 benchmark questions with ground-truth SQL.
2. Verifies each ground-truth query runs correctly against your data.
3. Pushes the questions to your primary Genie space via the API.

**Before you start:** Run notebooks **02** (data) and **03** (spaces).

**Compute:** Serverless.

## Compute

Use **Serverless**. Catalog and schema come from **widgets** only.


## Tips for writing good benchmarks

Benchmarks are the **unit tests of your Genie space**. They let you measure whether Genie answers correctly — and catch regressions when you change instructions or models.

- Ask for **one clear number or small result** per question (easy to score automatically).
- Compute expected answers in **SQL from the same tables** Genie uses — no manual guesswork.
- Mix **counts, averages, and filters** (e.g. OEE, downtime, defects).
- Use questions **real users would ask**; update benchmarks when the schema changes.
- In notebook 07 (compare), allow a small tolerance for rounding on ratio queries.
- The **curated SQL examples** in notebook 03 teach Genie *style*; benchmarks here track *correctness* over time.
- If the API step errors, you can add the same questions under **Genie Evaluation / Benchmarks** in the UI.

## Load config and print Genie space links

Reads the space IDs saved by notebook 03 and prints clickable links. After this notebook pushes benchmarks, open the **primary space** link to try them out.

In [ ]:
%run ./00_workshop_config

In [ ]:
import re
import json
import requests
import time
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

host = w.config.host.rstrip("/")
headers = {**w.config.authenticate(), "Content-Type": "application/json"}

def genie_ui_room_url(space_id):
    m = re.search(r"adb-(\d+)\.", host)
    o = m.group(1) if m else ""
    q = f"?o={o}" if o else ""
    return f"{host}/genie/rooms/{space_id}{q}"

_cfg = spark.table(full_table("workshop_config")).toPandas().set_index("key")["value"].to_dict()
PRIMARY_SPACE_ID = _cfg.get("genie_space_id", "")
BLANK_SPACE_ID = _cfg.get("genie_space_id_blank", "")
NO_EVALS_SPACE_ID = _cfg.get("genie_space_id_configured_no_evals", "")

if not PRIMARY_SPACE_ID:
    raise RuntimeError("genie_space_id not found in workshop_config. Run notebook 03 first.")

print("Your Genie spaces (click to open):")
print(f"  Configured: {genie_ui_room_url(PRIMARY_SPACE_ID)}")
if NO_EVALS_SPACE_ID:
    print(f"  No Examples: {genie_ui_room_url(NO_EVALS_SPACE_ID)}")
if BLANK_SPACE_ID:
    print(f"  Blank (no instructions):  {genie_ui_room_url(BLANK_SPACE_ID)}")
print()
print("Benchmarks will be pushed to the PRIMARY space.")
print("After this notebook completes, open the primary space link above")
print("and try asking one of the benchmark questions to verify it works.")

## Define benchmark suite

16 benchmark questions across two tiers:

**Core questions (Q1–Q12):** Counts, sums, filters, and joins that any well-configured space should get right.

**Advanced questions (Q13–Q16):** These are harder — they require multi-table joins and domain logic that Genie is more likely to get right when it has curated SQL examples to learn from.

| # | Question | What makes it hard |
|---|----------|--------------------|
| Q13 | Scrap rate by day for Michigan (last 5 days) | State join + date filter + ratio |
| Q14 | Plants with Maintenance-status lines | Status filter + join |
| Q15 | Lines with >5% historical defect rate | CASE/HAVING subquery |
| Q16 | Best-quality shift in Dec 2024 | events → operators join + aggregation |

In notebook **07**, you’ll run these same questions against both the "with examples" and "without examples" spaces to see the difference curated examples make.

In [ ]:
benchmarks_v1 = [
    {
        "q": "How many distinct production lines had at least one defect_detected event in calendar year 2024?",
        "sql": f"""SELECT CAST(COUNT(DISTINCT production_line_id) AS BIGINT) AS answer FROM {fqn}.production_events WHERE event_type = 'defect_detected' AND YEAR(CAST(event_date AS DATE)) = 2024""",
        "gt": f"SELECT CAST(COUNT(DISTINCT production_line_id) AS BIGINT) FROM {fqn}.production_events WHERE event_type = 'defect_detected' AND YEAR(CAST(event_date AS DATE)) = 2024",
    },
    {
        "q": "What is the sum of downtime_minutes from quality_metrics_daily for all rows in 2024 where the plant is in Michigan (join to plants by state)?",
        "sql": f"""SELECT CAST(ROUND(SUM(q.downtime_minutes), 0) AS BIGINT) AS answer FROM {fqn}.quality_metrics_daily q JOIN {fqn}.plants p ON q.plant_id = p.plant_id WHERE p.state = 'Michigan' AND YEAR(CAST(q.date AS DATE)) = 2024""",
        "gt": f"SELECT CAST(ROUND(SUM(q.downtime_minutes), 0) AS BIGINT) FROM {fqn}.quality_metrics_daily q JOIN {fqn}.plants p ON q.plant_id = p.plant_id WHERE p.state = 'Michigan' AND YEAR(CAST(q.date AS DATE)) = 2024",
    },
    {
        "q": "What is the total units_produced summed from quality_metrics_daily for January through March 2024 (Q1) only?",
        "sql": f"""SELECT CAST(SUM(q.units_produced) AS BIGINT) AS answer FROM {fqn}.quality_metrics_daily q WHERE CAST(q.date AS DATE) >= DATE '2024-01-01' AND CAST(q.date AS DATE) < DATE '2024-04-01'""",
        "gt": f"SELECT CAST(SUM(q.units_produced) AS BIGINT) FROM {fqn}.quality_metrics_daily q WHERE CAST(q.date AS DATE) >= DATE '2024-01-01' AND CAST(q.date AS DATE) < DATE '2024-04-01'",
    },
    {
        "q": "How many production lines had 500 or more unit_produced events in calendar year 2024?",
        "sql": f"""SELECT CAST(COUNT(*) AS BIGINT) AS answer FROM ( SELECT production_line_id FROM {fqn}.production_events WHERE event_type = 'unit_produced' AND YEAR(CAST(event_date AS DATE)) = 2024 GROUP BY production_line_id HAVING COUNT(*) >= 500 ) t""",
        "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM ( SELECT production_line_id FROM {fqn}.production_events WHERE event_type = 'unit_produced' AND YEAR(CAST(event_date AS DATE)) = 2024 GROUP BY production_line_id HAVING COUNT(*) >= 500 ) t",
    },
    {
        "q": "How many safety incidents are both severity Critical and category Equipment Malfunction?",
        "sql": f"""SELECT CAST(COUNT(*) AS BIGINT) AS answer FROM {fqn}.safety_incidents WHERE severity = 'Critical' AND category = 'Equipment Malfunction'""",
        "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM {fqn}.safety_incidents WHERE severity = 'Critical' AND category = 'Equipment Malfunction'",
    },
    {
        "q": "How many operators work at plants located in Michigan (use operators.plant_id joined to plants)?",
        "sql": f"""SELECT CAST(COUNT(*) AS BIGINT) AS answer FROM {fqn}.operators o JOIN {fqn}.plants p ON o.plant_id = p.plant_id WHERE p.state = 'Michigan'""",
        "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM {fqn}.operators o JOIN {fqn}.plants p ON o.plant_id = p.plant_id WHERE p.state = 'Michigan'",
    },
    {
        "q": "For calendar year 2024 quality_metrics_daily only, what is the overall defect rate in percent rounded to a whole number: 100 times the sum of defects_found divided by the sum of units_produced?",
        "sql": f"""SELECT CAST(ROUND(100.0 * SUM(defects_found) / SUM(units_produced), 0) AS BIGINT) AS answer FROM {fqn}.quality_metrics_daily WHERE YEAR(CAST(date AS DATE)) = 2024""",
        "gt": f"SELECT CAST(ROUND(100.0 * SUM(defects_found) / SUM(units_produced), 0) AS BIGINT) FROM {fqn}.quality_metrics_daily WHERE YEAR(CAST(date AS DATE)) = 2024",
    },
    {
        "q": "How many production events in 2024 have event_type inspection_passed?",
        "sql": f"""SELECT CAST(COUNT(*) AS BIGINT) AS answer FROM {fqn}.production_events WHERE event_type = 'inspection_passed' AND YEAR(CAST(event_date AS DATE)) = 2024""",
        "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM {fqn}.production_events WHERE event_type = 'inspection_passed' AND YEAR(CAST(event_date AS DATE)) = 2024",
    },
    {
        "q": "How many quality_metrics_daily rows in 2024 have oee_score strictly below 0.70?",
        "sql": f"""SELECT CAST(COUNT(*) AS BIGINT) AS answer FROM {fqn}.quality_metrics_daily WHERE YEAR(CAST(date AS DATE)) = 2024 AND oee_score < 0.70""",
        "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM {fqn}.quality_metrics_daily WHERE YEAR(CAST(date AS DATE)) = 2024 AND oee_score < 0.70",
    },
    {
        "q": "What is the sum of employee_count for all plants in Texas?",
        "sql": f"""SELECT CAST(COALESCE(SUM(p.employee_count), 0) AS BIGINT) AS answer FROM {fqn}.plants p WHERE p.state = 'Texas'""",
        "gt": f"SELECT CAST(COALESCE(SUM(p.employee_count), 0) AS BIGINT) FROM {fqn}.plants p WHERE p.state = 'Texas'",
    },
    {
        "q": "How many equipment_feedback rows are for production lines at the EV Battery Innovation Center (join feedback to production_lines and match that plant)?",
        "sql": f"""SELECT CAST(COUNT(*) AS BIGINT) AS answer FROM {fqn}.equipment_feedback f JOIN {fqn}.production_lines pl ON f.production_line_id = pl.line_id JOIN {fqn}.plants p ON pl.plant_id = p.plant_id WHERE p.plant_name = 'EV Battery Innovation Center'""",
        "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM {fqn}.equipment_feedback f JOIN {fqn}.production_lines pl ON f.production_line_id = pl.line_id JOIN {fqn}.plants p ON pl.plant_id = p.plant_id WHERE p.plant_name = 'EV Battery Innovation Center'",
    },
    {
        "q": "For plant P003 only and calendar year 2024, what is average OEE as a whole percent rounded to an integer (average oee_score times 100, then round)?",
        "sql": f"""SELECT CAST(ROUND(AVG(q.oee_score) * 100, 0) AS BIGINT) AS answer FROM {fqn}.quality_metrics_daily q WHERE q.plant_id = 'P003' AND YEAR(CAST(q.date AS DATE)) = 2024""",
        "gt": f"SELECT CAST(ROUND(AVG(q.oee_score) * 100, 0) AS BIGINT) FROM {fqn}.quality_metrics_daily q WHERE q.plant_id = 'P003' AND YEAR(CAST(q.date AS DATE)) = 2024",
    },
    # ── 4 advanced questions (Q13–Q16) (harder, expect different results between spaces) ──
    {
        "q": "What is the scrap rate by day for Michigan plants for the last 5 days of 2024? Show the total scrap_count divided by total units_produced as a percentage, rounded to 2 decimal places, for the 5 most recent days.",
        "sql": f"""SELECT CAST(ROUND(100.0 * SUM(q.scrap_count) / NULLIF(SUM(q.units_produced), 0), 2) AS DOUBLE) AS answer FROM {fqn}.quality_metrics_daily q JOIN {fqn}.plants p ON q.plant_id = p.plant_id WHERE p.state = 'Michigan' AND CAST(q.date AS DATE) >= DATE '2024-12-27' AND CAST(q.date AS DATE) <= DATE '2024-12-31'""",
        "gt": f"SELECT CAST(ROUND(100.0 * SUM(q.scrap_count) / NULLIF(SUM(q.units_produced), 0), 2) AS DOUBLE) FROM {fqn}.quality_metrics_daily q JOIN {fqn}.plants p ON q.plant_id = p.plant_id WHERE p.state = 'Michigan' AND CAST(q.date AS DATE) >= DATE '2024-12-27' AND CAST(q.date AS DATE) <= DATE '2024-12-31'",
    },
    {
        "q": "Which plants had a production line with status Maintenance show up in December 2024? Return the count of distinct plants.",
        "sql": f"""SELECT CAST(COUNT(DISTINCT p.plant_id) AS BIGINT) AS answer FROM {fqn}.production_lines pl JOIN {fqn}.plants p ON pl.plant_id = p.plant_id WHERE pl.status = 'Maintenance'""",
        "gt": f"SELECT CAST(COUNT(DISTINCT p.plant_id) AS BIGINT) FROM {fqn}.production_lines pl JOIN {fqn}.plants p ON pl.plant_id = p.plant_id WHERE pl.status = 'Maintenance'",
    },
    {
        "q": "How many production lines have a high historical defect rate — meaning more than 5% of their production_events in 2024 were defect_detected out of total unit_produced events?",
        "sql": f"""SELECT CAST(COUNT(*) AS BIGINT) AS answer FROM ( SELECT production_line_id, COUNT(CASE WHEN event_type = 'defect_detected' THEN 1 END) AS defects, COUNT(CASE WHEN event_type = 'unit_produced' THEN 1 END) AS produced FROM {fqn}.production_events WHERE YEAR(CAST(event_date AS DATE)) = 2024 GROUP BY production_line_id HAVING produced > 0 AND (100.0 * defects / produced) > 5.0 ) t""",
        "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM ( SELECT production_line_id, COUNT(CASE WHEN event_type = 'defect_detected' THEN 1 END) AS defects, COUNT(CASE WHEN event_type = 'unit_produced' THEN 1 END) AS produced FROM {fqn}.production_events WHERE YEAR(CAST(event_date AS DATE)) = 2024 GROUP BY production_line_id HAVING produced > 0 AND (100.0 * defects / produced) > 5.0 ) t",
    },
    {
        "q": "Which shift has the best quality in the last month of 2024? Return the count of defect_detected events for the shift with the fewest defects in December 2024 (join production_events to operators by operator_id to get the shift).",
        "sql": f"""SELECT CAST(MIN(defect_count) AS BIGINT) AS answer FROM ( SELECT o.shift, COUNT(*) AS defect_count FROM {fqn}.production_events pe JOIN {fqn}.operators o ON pe.operator_id = o.operator_id WHERE pe.event_type = 'defect_detected' AND CAST(pe.event_date AS DATE) >= DATE '2024-12-01' AND CAST(pe.event_date AS DATE) <= DATE '2024-12-31' GROUP BY o.shift ) t""",
        "gt": f"SELECT CAST(MIN(defect_count) AS BIGINT) FROM ( SELECT o.shift, COUNT(*) AS defect_count FROM {fqn}.production_events pe JOIN {fqn}.operators o ON pe.operator_id = o.operator_id WHERE pe.event_type = 'defect_detected' AND CAST(pe.event_date AS DATE) >= DATE '2024-12-01' AND CAST(pe.event_date AS DATE) <= DATE '2024-12-31' GROUP BY o.shift ) t",
    },
]
print(f"{len(benchmarks_v1)} benchmarks (v2 harder suite + 4 advanced questions (Q13–Q16))")

## Verify ground truth in Spark

Run each `gt` query once. Fix definitions before pushing to Genie if any fail.


In [ ]:
for i, b in enumerate(benchmarks_v1, 1):
    try:
        v = spark.sql(b["gt"]).collect()[0][0]
        print(f"OK Q{i}: {v}")
    except Exception as e:
        print(f"FAIL Q{i}: {e}")


## Push benchmarks to Genie (update evaluation rail)

Registers **BENCHMARK** curated questions on the **primary** space. If the data-rooms API errors, copy the printed Q→SQL into Genie manually.


In [ ]:
import uuid

# Step 1: Fetch the current space definition (including existing curated Q→SQL pairs)
get_resp = requests.get(
    f"{host}/api/2.0/genie/spaces/{PRIMARY_SPACE_ID}?include_serialized_space=true",
    headers=headers,
)
get_resp.raise_for_status()
existing = get_resp.json()
_ss = existing.get("serialized_space")
space_def = json.loads(_ss) if isinstance(_ss, str) else (_ss if isinstance(_ss, dict) else {})

# Step 2: Collect questions already in the space to avoid duplicates
existing_qs = {
    eq["question"][0] if isinstance(eq.get("question"), list) else eq.get("question", "")
    for eq in space_def.get("instructions", {}).get("example_question_sqls", [])
}

# Step 3: Build a list of new benchmark Q→SQL pairs not already in the space
new_curated = []
for b in benchmarks_v1:
    if b["q"] not in existing_qs:
        new_curated.append({
            "id": uuid.uuid4().hex,
            "question": [b["q"]],
            "sql": [b["sql"]],
        })

# Step 4: Append new pairs and PATCH the space definition
# IMPORTANT: example_question_sqls must be sorted by ID or the API rejects the request
if new_curated:
    eqs = space_def.setdefault("instructions", {}).setdefault("example_question_sqls", [])
    eqs.extend(new_curated)
    eqs.sort(key=lambda x: x.get("id", ""))
    patch_resp = requests.patch(
        f"{host}/api/2.0/genie/spaces/{PRIMARY_SPACE_ID}",
        headers=headers,
        json={"serialized_space": json.dumps(space_def)},
    )
    if patch_resp.status_code == 200:
        print(f"Pushed {len(new_curated)} new benchmark Q→SQL pairs to primary space {PRIMARY_SPACE_ID}")
    else:
        print(f"PATCH failed ({patch_resp.status_code}): {patch_resp.text[:400]}")
        print("Manual fallback: add the benchmark Q→SQL pairs in the Genie UI under Evaluation.")
else:
    print(f"All {len(benchmarks_v1)} benchmark questions already present in primary space.")


## Try it out


In [ ]:
print("Open your primary Genie space and try a few benchmark questions:")
print(f"  {genie_ui_room_url(PRIMARY_SPACE_ID)}")
print()
print("Sample questions to paste into the Genie chat:")
for i, b in enumerate(benchmarks_v1[:4], 1):
    print(f"  Q{i}: {b['q']}")
print()
print("Next: Notebook 05 — explore your Genie space interactively.")